<a href="https://colab.research.google.com/github/ThisumiWijesinghe/Evidence-Verified-RAG-Workflow-/blob/main/Main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install Ollama

In [1]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 102 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (1,131 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


# Start Ollama Server in background

In [3]:
import subprocess
import time

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

# Wait for the server to start
time.sleep(5)
print("Ollama server started, PID:", process.pid)


Ollama server started, PID: 2042


#Pull LIama 3.1

In [4]:
!ollama pull llama3.1:8b

#Pull the embedding model

In [5]:
!ollama pull nomic-embed-text

# Verify both are installed

In [6]:
!ollama list

NAME                       ID              SIZE      MODIFIED               
nomic-embed-text:latest    0a109f422b47    274 MB    Less than a second ago    
llama3.1:8b                46e0c10c039e    4.9 GB    5 seconds ago             


#Test generation works

In [7]:
import requests
response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.1:8b",
        "prompt": "Say 'setup successful' and nothing else.",
        "stream": False,
    }
)
print(response.json()["response"])

Setup successful.


#Install Langchain+dependencies for next steps

In [8]:
!pip install -q langchain langchain-community langchain-ollama faiss-cpu pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 83.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.1/405.1 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


Connect Langchain to Ollama

In [9]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.1:8b", temperature=0.2)
result = llm.invoke("In one sentence, what is Retrieval Augmented Generation?")
print(result)

Retrieval Augmented Generation (RAG) is a type of artificial intelligence model that combines the strengths of retrieval-based and generative models by first retrieving relevant information from a large database and then generating new text based on that information.


#PDF Ingestion + Chunking

## Upload a test PDF to colab

In [10]:
from google.colab import files

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print("Uploaded:",pdf_path)

Saving dl based detecting malicious packages in pypl.pdf to dl based detecting malicious packages in pypl.pdf
Saving On the Performance of Malware Detection.pdf to On the Performance of Malware Detection.pdf
Saving Quantum redy crypto.pdf to Quantum redy crypto.pdf
Uploaded: dl based detecting malicious packages in pypl.pdf


#Install the text-splitter package

In [11]:
!pip install -q langchain langchain-community langchain-ollama  langchain-text-splitters faiss-cpu pypdf

##Load the PDF with PyPDFLoader

In [12]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Loaded {len(pages)} pages")
print("Sample text from page 1:\n", pages[0].page_content[:500])

/tmp/ipykernel_818/2514516427.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 26 pages
Sample text from page 1:
 eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
Raja Jurdak
Queensland University of Technology
Brisbane, QLD, Australia
Chadni Islam
Edith Cowan University
Joondalup, WA, Australia
Abu Bakar Siddique Mahi
Research Graduate School, BUBT
Dhaka, Bangladesh
Gowri Ramachandran
Queensland University of Technology
Brisbane, QLD, Australia
Abstract
The s


##Chunk the text with RecursiveChracterTextSplitter

In [13]:
!pip install -q -U langchain-text-splitters


In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
print("import successful!")

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(pages)

print(f"Split into {len(chunks)} chunks")
print("Sample text from chunk 1:\n", chunks[0].page_content[:300])

import successful!
Split into 148 chunks
Sample text from chunk 1:
 eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
Raja Jurdak
Queensland University of Technology
Brisbane, QLD, Australia
Chadni Islam
Edith Cowan Uni


##Quick sanity check -- Inspect a few chunks

In [15]:
for i in range(min(3, len(chunks))):
  print(f"---chunk {i} ---")
  print(chunks[i].page_content[:200])
  print("Metadata:", chunks[i].metadata)
  print()

---chunk 0 ---
eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
R
Metadata: {'producer': 'pikepdf 8.15.1', 'creator': 'arXiv GenPDF (tex2pdf:a6404ea)', 'creationdate': '2026-04-30T00:20:09+00:00', 'author': 'Sk Tanzir Mehedi; Raja Jurdak; Chadni Islam; Abu Bakar Siddique Mahi; Gowri Ramachandran', 'doi': 'https://doi.org/10.48550/arXiv.2604.26219', 'keywords': 'Deep learning, malicious detection, software supply chain security, dynamic analysis, stability, explainable AI.', 'license': 'http://creativecommons.org/licenses/by/4.0/', 'moddate': '2026-04-30T00:20:09+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.28 (TeX Live 2025) kpathsea version 6.4.1', 'title': 'eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem', 'trapped': '/False', 'arxivid': 

#Embeddings & FAISS vector store , test retrieval

##Connect the nomic-embed-text

In [16]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

print("Embedding model connected succesfully!")

Embedding model connected succesfully!


##Test the embedding model

In [17]:
test_vector = embeddings.embed_query(
    "What is malicious package detection?"
)

print("Embedding dimension:", len(test_vector))
print("first 10 values:", test_vector[:10])

Embedding dimension: 768
first 10 values: [0.07384906, 0.020637663, -0.15453403, -0.018535705, 0.06982891, -0.04924275, -0.005812537, 0.03157499, -0.0047331247, -0.0016610859]


## Create the FAISS vector database

In [18]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print("FAISS Vector database created successfully!")
print("Number of chunks stored:", len(chunks))

FAISS Vector database created successfully!
Number of chunks stored: 148


#FAISS Test retrieval

In [19]:

# ============================================================
# FIX: RESTART OLLAMA SERVER
# ============================================================

import subprocess
import time
import requests

# Start Ollama without blocking the notebook
ollama_log = open("/tmp/ollama_server.log", "a")

process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=ollama_log,
    stderr=subprocess.STDOUT
)

# Wait until Ollama responds
server_ready = False

for attempt in range(20):
    try:
        response = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=3
        )

        if response.status_code == 200:
            server_ready = True
            break

    except requests.exceptions.RequestException:
        pass

    time.sleep(1)

if server_ready:
    print("Ollama server started successfully!")
    print("Available models:")
    print(requests.get(
        "http://127.0.0.1:11434/api/tags"
    ).json())
else:
    print("Ollama server failed to start.")
    print("Check /tmp/ollama_server.log")


Ollama server started successfully!
Available models:
{'models': [{'name': 'nomic-embed-text:latest', 'model': 'nomic-embed-text:latest', 'modified_at': '2026-10-20T05:14:32.614089983Z', 'size': 274302450, 'digest': '0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'nomic-bert', 'families': ['nomic-bert'], 'parameter_size': '137M', 'quantization_level': 'F16', 'context_length': 2048, 'embedding_length': 768, 'runner': 'ggml'}, 'capabilities': ['embedding']}, {'name': 'llama3.1:8b', 'model': 'llama3.1:8b', 'modified_at': '2026-10-20T05:14:27.577737461Z', 'size': 4920753328, 'digest': '46e0c10c039e019119339687c3c1757cc81b9da49709a3b3924863ba87ca666e', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '8.0B', 'quantization_level': 'Q4_K_M', 'context_length': 131072, 'embedding_length': 4096, 'runner': 'ggml'}, 'capabilities': ['completion', 'tools']}]}


In [23]:


# ============================================================
# FIX: START OLLAMA SERVER SAFELY
# ============================================================

import subprocess
import requests
import time

OLLAMA_URL = "http://127.0.0.1:11434"

def check_ollama():
    try:
        response = requests.get(
            f"{OLLAMA_URL}/api/tags",
            timeout=5
        )
        return response.status_code == 200
    except requests.exceptions.RequestException:
        return False

if check_ollama():
    print("Ollama server is already running!")

else:
    print("Starting Ollama server...")

    # Write logs to a file instead of a PIPE.
    ollama_log = open("/tmp/ollama_server.log", "a")

    process = subprocess.Popen(
        ["ollama", "serve"],
        stdout=ollama_log,
        stderr=subprocess.STDOUT
    )

    for attempt in range(20):
        if check_ollama():
            print("Ollama server started successfully!")
            break
        time.sleep(1)
    else:
        print("Server failed to start.")
        print("Check /tmp/ollama_server.log")

print("Server ready:", check_ollama())


Starting Ollama server...
Ollama server started successfully!
Server ready: True


In [24]:

# ============================================================
# TEST EMBEDDING MODEL
# ============================================================

import time

start = time.time()

test_embedding = embeddings.embed_query(
    "What problem does eDySec aim to solve?"
)

print("Embedding successful!")
print("Embedding dimension:", len(test_embedding))
print("Time:", round(time.time() - start, 2), "seconds")


Embedding successful!
Embedding dimension: 768
Time: 13.23 seconds


In [25]:

# ============================================================
# TEST FAISS RETRIEVAL
# ============================================================

question = "What problem does eDySec aim to solve?"

retrieved_docs = vectorstore.similarity_search(
    question,
    k=4
)

print("Retrieved chunks:", len(retrieved_docs))

for i, doc in enumerate(retrieved_docs):
    print(f"\n{'='*60}")
    print(f"Retrieved Chunk {i+1}")
    print("Page:", doc.metadata.get("page_label", "Unknown"))
    print("="*60)
    print(doc.page_content[:500])


Retrieved chunks: 4

Retrieved Chunk 1
Page: 6
Conference Name, Month Year, City, Country Tanzir et al.
between the selected feature subset and the model’s learned be-
havior is assessed throughC=F ∗∩Feat(R 𝑒𝑥𝑝), where a larger
overlap indicates stronger agreement between the selected features
and the features that most strongly influence model predictions.
Algorithm 1 presents the pseudo-code for the proposed eDySec
framework workflow.
4 eDySec Design and Setup
This section describes the experimental design of eDySec, including
the computat

Retrieved Chunk 2
Page: 13
eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem Conference Name, Month Year, City, Country
References
[1] Hubert Baniecki and Przemyslaw Biecek. 2024. Adversarial Attacks and Defenses
in Explainable Artificial Intelligence: A Survey.Information Fusion107 (2024),
102303. doi:10.1016/j.inffus.2024.102303
[2] Alessio Benavoli, Giorgio Corani, Janez Demš

# Multi Query Generartion

In [26]:

# ============================================================
# ADVANCED RAG - IMPROVED MULTI-QUERY GENERATION
# ============================================================

def generate_search_queries(question, llm, vectorstore):

    # Retrieve initial context from the uploaded paper
    initial_docs = vectorstore.similarity_search(
        question,
        k=4
    )

    context = "\n\n".join(
        doc.page_content[:700]
        for doc in initial_docs
    )

    prompt = f"""
You are an academic research retrieval assistant.

Generate exactly 3 alternative search queries.

ORIGINAL QUESTION:
{question}

DOCUMENT CONTEXT:
{context}

RULES:
1. Preserve the original question's meaning.
2. Use only terminology supported by the document context.
3. Do not introduce unrelated concepts.
4. Do not answer the question.
5. Do not number the queries.
6. Return exactly 3 queries, one per line.

SEARCH QUERIES:
"""

    response = llm.invoke(prompt)

    queries = [
        line.strip().lstrip("-").strip()
        for line in response.split("\n")
        if line.strip()
    ]

    return [question] + queries[:3]


In [27]:

# ============================================================
# TEST IMPROVED MULTI-QUERY GENERATION
# ============================================================

question = "What problem does eDySec aim to solve?"

search_queries = generate_search_queries(
    question,
    llm,
    vectorstore
)

print("Improved Search Queries:")
print("=" * 60)

for i, query in enumerate(search_queries):
    print(f"{i+1}. {query}")


Improved Search Queries:
1. What problem does eDySec aim to solve?
2. What problem does eDySec aim to solve in the PyPI ecosystem through explainable dynamic analysis?
3. What issue does eDySec address in the context of malicious package detection using deep learning and feature selection?
4. What challenge does eDySec seek to overcome in the PyPI ecosystem through the integration of explainability techniques and deep learning?


In [28]:

# ============================================================
# ADVANCED RAG - STEP 2: MULTI-QUERY FAISS RETRIEVAL
# ============================================================

def multi_query_retrieve(search_queries, vectorstore, k=6):
    """
    Retrieve document chunks using multiple search queries
    and remove duplicate results.
    """

    all_docs = []

    # Retrieve relevant chunks for each query
    for i, query in enumerate(search_queries):

        print(f"Searching query {i+1}/{len(search_queries)}...")

        docs = vectorstore.similarity_search(
            query,
            k=k
        )

        all_docs.extend(docs)

    print("\nTotal retrieved chunks:", len(all_docs))

    # Remove duplicate chunks
    unique_docs = []
    seen = set()

    for doc in all_docs:

        key = (
            doc.metadata.get("source"),
            doc.metadata.get("page"),
            doc.page_content
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    print("Unique chunks after deduplication:", len(unique_docs))

    return unique_docs


In [29]:

# ============================================================
# TEST MULTI-QUERY RETRIEVAL
# ============================================================

candidate_docs = multi_query_retrieve(
    search_queries,
    vectorstore,
    k=6
)

print("\n" + "=" * 60)
print("MULTI-QUERY RETRIEVAL RESULTS")
print("=" * 60)

for i, doc in enumerate(candidate_docs):

    print(f"\n--- Candidate {i+1} ---")
    print("Page:", doc.metadata.get("page_label", "Unknown"))
    print(doc.page_content[:300])


Searching query 1/4...
Searching query 2/4...
Searching query 3/4...
Searching query 4/4...

Total retrieved chunks: 24
Unique chunks after deduplication: 15

MULTI-QUERY RETRIEVAL RESULTS

--- Candidate 1 ---
Page: 6
Conference Name, Month Year, City, Country Tanzir et al.
between the selected feature subset and the model’s learned be-
havior is assessed throughC=F ∗∩Feat(R 𝑒𝑥𝑝), where a larger
overlap indicates stronger agreement between the selected features
and the features that most strongly influence model 

--- Candidate 2 ---
Page: 13
eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem Conference Name, Month Year, City, Country
References
[1] Hubert Baniecki and Przemyslaw Biecek. 2024. Adversarial Attacks and Defenses
in Explainable Artificial Intelligence: A Su

--- Candidate 3 ---
Page: 15
eDySec: A Deep Learning-based Explainable Dynamic Analysis Framework for Detecting Malicious Packages in PyPI Ecosystem 

#Cross - encoder reranking

Install the reranking model dependencies

In [30]:

# ============================================================
# ADVANCED RAG - STEP 3A: INSTALL RERANKER
# ============================================================

!pip install -q sentence-transformers


Load cross encoder ranker

In [31]:

# ============================================================
# ADVANCED RAG - STEP 3B: LOAD CROSS-ENCODER
# ============================================================

from sentence_transformers import CrossEncoder
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2",
    device=device
)

print("Reranker loaded successfully!")
print("Device:", device)


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Reranker loaded successfully!
Device: cuda


Create reranking function

In [32]:


# ============================================================
# ADVANCED RAG - STEP 3C: RERANK RETRIEVED CHUNKS
# ============================================================

def rerank_documents(question, candidate_docs, reranker, top_n=5):
    """
    Rerank retrieved chunks using a Cross-Encoder model.
    Return the top-ranked documents and their scores.
    """

    if not candidate_docs:
        return []

    # Create question-document pairs
    pairs = [
        (question, doc.page_content)
        for doc in candidate_docs
    ]

    # Calculate relevance scores
    scores = reranker.predict(pairs)

    # Combine documents with scores
    scored_docs = list(zip(candidate_docs, scores))

    # Sort by relevance (highest first)
    scored_docs.sort(
        key=lambda item: float(item[1]),
        reverse=True
    )

    # Select top-ranked chunks
    return scored_docs[:top_n]



Test Cross-Encoder Reranking

In [33]:

# ============================================================
# ADVANCED RAG - STEP 3D: TEST RERANKING
# ============================================================

question = "What problem does eDySec aim to solve?"

reranked_docs = rerank_documents(
    question,
    candidate_docs,
    reranker,
    top_n=5
)

print("=" * 65)
print("TOP 5 RERANKED EVIDENCE CHUNKS")
print("=" * 65)

for i, (doc, score) in enumerate(reranked_docs):

    print(f"\n{'='*65}")
    print(f"Rank: {i+1}")
    print(f"Page: {doc.metadata.get('page_label', 'Unknown')}")
    print(f"Relevance score: {float(score):.4f}")
    print("=" * 65)

    print(doc.page_content[:600])


TOP 5 RERANKED EVIDENCE CHUNKS

Rank: 1
Page: 2
Relevance score: 1.4210
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per package.
This study is structured as follows: Section 2 reviews existing
methods for malicious package d

Rank: 2
Page: 9
Relevance score: 0.9496
better trade-off between attack sensitivity and false-alarm control.
This pattern is particularly important in malicious package detec-
tion, where false negatives allow threats to evade detection and
false positives undermine the practical usability of the detector.
Beyond quantitativ

# Retrieval Quality Grading

Define retrieval grader

In [34]:

# ============================================================
# ADVANCED RAG - STEP 4: RETRIEVAL QUALITY GRADING
# ============================================================

def grade_retrieval(question, reranked_docs, llm):

    # Handle empty retrieval results.
    if not reranked_docs:
        return "INSUFFICIENT"

    # Prepare the top-ranked evidence for evaluation.
    context = "\n\n".join(
        f"Evidence {i+1}:\n{doc.page_content}"
        for i, (doc, score) in enumerate(reranked_docs)
    )

    # Ask the LLM to evaluate evidence sufficiency.
    prompt = f"""
You are an evidence relevance evaluator for academic research.

USER QUESTION:
{question}

RETRIEVED EVIDENCE:
{context}

TASK:
Determine whether the retrieved evidence contains enough
relevant information to answer the user's question.

RULES:
1. Evaluate only the provided evidence.
2. Do not use external knowledge.
3. Do not answer the question.
4. Return exactly one word:
SUFFICIENT
or
INSUFFICIENT

DECISION:
"""

    response = llm.invoke(prompt).strip().upper()

    # Accept only a valid decision.
    if response == "SUFFICIENT":
        return "SUFFICIENT"

    return "INSUFFICIENT"


In [35]:

# ============================================================
# STEP 4B: TEST RETRIEVAL QUALITY GRADING
# ============================================================

question = "What problem does eDySec aim to solve?"

retrieval_status = grade_retrieval(
    question,
    reranked_docs,
    llm
)

print("=" * 60)
print("RETRIEVAL QUALITY ASSESSMENT")
print("=" * 60)

print("Question:", question)
print("Evidence status:", retrieval_status)

if retrieval_status == "SUFFICIENT":
    print("Decision: Proceed to answer generation.")
else:
    print("Decision: Additional evidence retrieval required.")


RETRIEVAL QUALITY ASSESSMENT
Question: What problem does eDySec aim to solve?
Evidence status: INSUFFICIENT
Decision: Additional evidence retrieval required.


# Rertival quality grader debugging

In [36]:

# ============================================================
# STEP 5A: DEBUG RETRIEVAL QUALITY GRADER
# ============================================================

# Prepare evidence from the reranked documents.
context = "\n\n".join(
    f"Evidence {i+1} (Page {doc.metadata.get('page_label', 'Unknown')}):\n"
    f"{doc.page_content}"
    for i, (doc, score) in enumerate(reranked_docs)
)

debug_prompt = f"""
You are evaluating evidence for scientific question answering.

QUESTION:
{question}

EVIDENCE:
{context}

TASK:
Determine whether the evidence is sufficient to answer
the question using only the supplied passages.

Explain briefly:
1. What relevant information is available?
2. What information is missing, if any?
3. Can the question be answered from this evidence?

Finish with exactly one of these labels:
FINAL_DECISION: SUFFICIENT
FINAL_DECISION: INSUFFICIENT
"""

response = llm.invoke(debug_prompt)

print("=" * 60)
print("RETRIEVAL GRADER DEBUG RESULT")
print("=" * 60)
print(response)


RETRIEVAL GRADER DEBUG RESULT
1. Relevant information available:
The evidence provides information about the eDySec framework, its features, and its performance in detecting malicious packages. Specifically, it mentions:
	* The incorporation of model stability analysis and explainable AI techniques
	* The reduction of feature dimensionality by 52.78%
	* The lowering of false positives by 82% and false negatives by 79%
	* The improvement of accuracy by 3%
	* The maintenance of a low inference latency of 170ms per package
	* The use of deep learning-based formulation to capture complex interactions in high-dimensional data
2. Missing information:
The evidence does not provide a clear answer to the question of what problem eDySec aims to solve. While it mentions the detection of malicious packages, it does not explicitly state the problem that eDySec addresses.
3. Can the question be answered from this evidence?
No, the question cannot be answered from this evidence. The evidence provides

In [37]:

# ============================================================
# STEP 5B: INSPECT ORIGINAL PAPER PROBLEM STATEMENT
# ============================================================

print("=" * 65)
print("ORIGINAL PAPER: ABSTRACT AND INTRODUCTION")
print("=" * 65)

# Inspect the first two pages of the uploaded paper.
for page in pages[:2]:

    print("\n" + "=" * 65)
    print("PAGE:", page.metadata.get("page_label", "Unknown"))
    print("=" * 65)

    print(page.page_content[:5000])


ORIGINAL PAPER: ABSTRACT AND INTRODUCTION

PAGE: 1
eDySec: A Deep Learning-based Explainable Dynamic Analysis
Framework for Detecting Malicious Packages in PyPI Ecosystem
Sk Tanzir Mehedi∗
Queensland University of Technology
Brisbane, QLD, Australia
Raja Jurdak
Queensland University of Technology
Brisbane, QLD, Australia
Chadni Islam
Edith Cowan University
Joondalup, WA, Australia
Abu Bakar Siddique Mahi
Research Graduate School, BUBT
Dhaka, Bangladesh
Gowri Ramachandran
Queensland University of Technology
Brisbane, QLD, Australia
Abstract
The security of open-source software repositories is increasingly
threatened by next-gen software supply chain attacks. These at-
tacks include multiphase malware execution, remote access ac-
tivation, and dynamic payload generation. Traditional Machine
Learning (ML) detectors struggle to detect these attacks due to the
high-dimensional and sparse nature of dynamic behavioral data,
including system calls, network traffic, directory access patterns,
a

In [38]:

# ============================================================
# STEP 5C: CORRECTIVE RETRIEVAL FOR INSUFFICIENT EVIDENCE
# ============================================================

def corrective_retrieval(question, vectorstore, reranker, top_n=5):

    # Target missing information using alternative queries.
    corrective_queries = [
        question,
        "What limitations of existing malicious package detection methods does eDySec address?",
        "What challenges motivate the development of eDySec?",
        "What problems with dynamic behavioral data, model stability and explainability does eDySec solve?"
    ]

    all_docs = []

    # Retrieve evidence for each query.
    for query in corrective_queries:
        docs = vectorstore.similarity_search(query, k=6)
        all_docs.extend(docs)

    # Remove duplicate documents.
    unique_docs = []
    seen = set()

    for doc in all_docs:
        key = (
            doc.metadata.get("source"),
            doc.metadata.get("page"),
            doc.page_content
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    # Rerank the retrieved evidence.
    corrected_docs = rerank_documents(
        question,
        unique_docs,
        reranker,
        top_n=top_n
    )

    print("Corrective queries:", len(corrective_queries))
    print("Unique evidence chunks:", len(unique_docs))

    return corrected_docs


# Run corrective retrieval only when evidence is insufficient.
if retrieval_status == "INSUFFICIENT":

    corrected_docs = corrective_retrieval(
        question,
        vectorstore,
        reranker,
        top_n=5
    )

    print("\nCORRECTIVE RETRIEVAL RESULTS")

    for i, (doc, score) in enumerate(corrected_docs):
        print(f"\nRank {i+1}")
        print("Page:", doc.metadata.get("page_label"))
        print("Score:", round(float(score), 4))
        print(doc.page_content[:700])

else:
    corrected_docs = reranked_docs
    print("Original evidence accepted.")


Corrective queries: 4
Unique evidence chunks: 16

CORRECTIVE RETRIEVAL RESULTS

Rank 1
Page: 2
Score: 1.421
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per package.
This study is structured as follows: Section 2 reviews existing
methods for malicious package detection. Section 3 presents the
eDySec framework, and Section 4 describes dataset, experimental
des

Rank 2
Page: 2
Score: 1.3178
ment, and transparent interpretation of model decisions, creates a
dangerous illusion of security and highlights the need for a more
comprehensive analytical fra

In [39]:

# ============================================================
# STEP 5D: RE-EVALUATE CORRECTED EVIDENCE
# ============================================================

print("=" * 60)
print("CORRECTIVE RETRIEVAL EVALUATION")
print("=" * 60)

# Evaluate the corrected evidence using the existing grader.
corrected_status = grade_retrieval(
    question,
    corrected_docs,
    llm
)

print("Original evidence status:", retrieval_status)
print("Corrected evidence status:", corrected_status)

if corrected_status == "SUFFICIENT":
    print("Decision: Proceed to grounded answer generation.")
else:
    print("Decision: Evidence remains insufficient.")
    print("Next action: Inspect evidence coverage or abstain.")


CORRECTIVE RETRIEVAL EVALUATION
Original evidence status: INSUFFICIENT
Corrected evidence status: INSUFFICIENT
Decision: Evidence remains insufficient.
Next action: Inspect evidence coverage or abstain.


In [40]:

# ============================================================
# STEP 5E: DIAGNOSE EVIDENCE COVERAGE
# ============================================================

# Diagnostic phrases from the paper's problem statement.
# These are used only to inspect the current pipeline.
problem_phrases = [
    "Traditional Machine Learning",
    "high-dimensional and sparse",
    "false positives and false negatives",
    "explainability analysis",
    "model stability"
]

def find_problem_evidence(documents):
    matches = []

    for doc in documents:
        content = doc.page_content.lower()

        found = [
            phrase
            for phrase in problem_phrases
            if phrase.lower() in content
        ]

        if found:
            matches.append((doc, found))

    return matches


# 1. Search all indexed chunks.
corpus_matches = find_problem_evidence(chunks)

# 2. Search corrective retrieval candidates.
# Reconstruct the candidate pool for diagnosis.
corrective_queries = [
    question,
    "What limitations of existing malicious package detection methods does eDySec address?",
    "What challenges motivate the development of eDySec?",
    "What problems with dynamic behavioral data, model stability and explainability does eDySec solve?"
]

candidate_pool = []

for query in corrective_queries:
    candidate_pool.extend(
        vectorstore.similarity_search(query, k=6)
    )

candidate_matches = find_problem_evidence(candidate_pool)

# 3. Search final top-5 reranked chunks.
top_docs = [doc for doc, score in corrected_docs]
top_matches = find_problem_evidence(top_docs)

print("=" * 60)
print("EVIDENCE COVERAGE DIAGNOSIS")
print("=" * 60)

print("Matching chunks in full corpus:", len(corpus_matches))
print("Matching entries in retrieval pool:", len(candidate_matches))
print("Matching chunks in final top 5:", len(top_matches))

print("\nFINAL TOP-5 MATCH DETAILS")

for i, (doc, phrases) in enumerate(top_matches, 1):
    print(f"\nMatch {i}")
    print("Page:", doc.metadata.get("page_label"))
    print("Matched phrases:", phrases)


EVIDENCE COVERAGE DIAGNOSIS
Matching chunks in full corpus: 18
Matching entries in retrieval pool: 4
Matching chunks in final top 5: 1

FINAL TOP-5 MATCH DETAILS

Match 1
Page: 1
Matched phrases: ['model stability']


In [41]:

# ============================================================
# STEP 5F: DIVERSITY-AWARE EVIDENCE SELECTION
# ============================================================

def select_diverse_evidence(
    question,
    candidate_docs,
    reranker,
    top_n=5,
    relevance_weight=0.7
):
    """
    Select relevant evidence while reducing redundancy.

    This method uses Cross-Encoder relevance scores and
    embedding similarity to select diverse chunks.
    """

    if not candidate_docs:
        return []

    # Score all candidate documents for relevance.
    pairs = [
        (question, doc.page_content)
        for doc in candidate_docs
    ]

    scores = reranker.predict(pairs)

    # Convert scores to a comparable 0-1 scale.
    import numpy as np

    scores = np.asarray(scores, dtype=float)

    score_range = scores.max() - scores.min()

    if score_range > 0:
        normalized_scores = (
            scores - scores.min()
        ) / score_range
    else:
        normalized_scores = np.ones_like(scores)

    # Embed candidate documents for similarity comparison.
    texts = [doc.page_content for doc in candidate_docs]
    vectors = np.asarray(
        embeddings.embed_documents(texts),
        dtype=float
    )

    # Normalize vectors for cosine similarity.
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    vectors = vectors / np.maximum(norms, 1e-12)

    selected = []
    remaining = list(range(len(candidate_docs)))

    while remaining and len(selected) < top_n:

        best_idx = None
        best_score = float("-inf")

        for idx in remaining:

            relevance = normalized_scores[idx]

            if selected:
                redundancy = max(
                    float(np.dot(vectors[idx], vectors[j]))
                    for j in selected
                )
            else:
                redundancy = 0.0

            combined_score = (
                relevance_weight * relevance
                - (1 - relevance_weight) * redundancy
            )

            if combined_score > best_score:
                best_score = combined_score
                best_idx = idx

        selected.append(best_idx)
        remaining.remove(best_idx)

    return [
        (candidate_docs[i], float(scores[i]))
        for i in selected
    ]


# Remove duplicate candidates from corrective retrieval.
unique_candidates = []
seen = set()

for doc in candidate_pool:
    key = (
        doc.metadata.get("source"),
        doc.metadata.get("page"),
        doc.page_content
    )

    if key not in seen:
        seen.add(key)
        unique_candidates.append(doc)


diverse_docs = select_diverse_evidence(
    question,
    unique_candidates,
    reranker,
    top_n=5
)

print("=" * 60)
print("DIVERSITY-AWARE EVIDENCE RESULTS")
print("=" * 60)

for i, (doc, score) in enumerate(diverse_docs, 1):
    print(f"\nRank {i}")
    print("Page:", doc.metadata.get("page_label"))
    print("Relevance score:", round(score, 4))
    print(doc.page_content[:500])


DIVERSITY-AWARE EVIDENCE RESULTS

Rank 1
Page: 2
Relevance score: 1.421
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per pack

Rank 2
Page: 15
Relevance score: 1.2273
of the eDySec pipeline, including data handling, preprocessing, fea-
ture extraction, model training, and evaluation, were performed
in isolated and controlled environments to reduce operational risk
and ensure safe experimentation. The dataset was used exclusively
for research and defensive purposes to advance malicious pack-
age detection in software supply chains. No personal, pri

In [42]:

# ============================================================
# STEP 5G: DIAGNOSE PROBLEM-STATEMENT RETRIEVAL
# ============================================================

diagnostic_queries = [
    "Traditional machine learning detectors struggle to detect malicious packages",
    "High-dimensional sparse dynamic behavioral data challenges",
    "Limitations of existing methods for malicious PyPI package detection"
]

for i, query in enumerate(diagnostic_queries, 1):

    print("\n" + "=" * 65)
    print(f"DIAGNOSTIC QUERY {i}: {query}")
    print("=" * 65)

    docs = vectorstore.similarity_search(query, k=5)

    for j, doc in enumerate(docs, 1):

        print(f"\nResult {j}")
        print("Page:", doc.metadata.get("page_label", "Unknown"))
        print(doc.page_content[:450])



DIAGNOSTIC QUERY 1: Traditional machine learning detectors struggle to detect malicious packages

Result 1
Page: 1
Learning (ML) detectors struggle to detect these attacks due to the
high-dimensional and sparse nature of dynamic behavioral data,
including system calls, network traffic, directory access patterns,
and dependency logs. As a result, these data characteristics degrade
the performance, stability, and explainability of ML models. These
challenges have made Deep Learning (DL) a promising alternative,
given its success across various domains and its p

Result 2
Page: 1
malicious package detection. Additionally, model stability analysis
and explainable AI techniques are incorporated into the detection
pipeline to enable stable, and transparent interpretations of model
decisions. Experimental results demonstrate that eDySec signifi-
cantly outperforms the state-of-the-art frameworks. Specifically,
it halves feature dimensionality while lowering false positives by
82% and false n

In [43]:

# ============================================================
# STEP 5H: AUTOMATED CORRECTIVE QUERY GENERATION
# ============================================================

def generate_corrective_queries(question, evidence_docs, llm):

    # Use retrieved evidence to identify missing information.
    context = "\n\n".join(
        doc.page_content[:700]
        for doc, score in evidence_docs[:5]
    )

    prompt = f"""
You are a scientific information retrieval assistant.

USER QUESTION:
{question}

CURRENT RETRIEVED EVIDENCE:
{context}

The evidence has been marked as insufficient.

Generate exactly 3 alternative search queries to retrieve
the missing information needed to answer the user question.

INSTRUCTIONS:
- Focus on the underlying research problem and motivation.
- Use terminology supported by the evidence.
- Do not invent facts or assume missing information is true.
- Avoid searching for performance results unless required.
- Do not answer the question.
- Output only 3 search queries, one per line.
- Do not number the queries.

SEARCH QUERIES:
"""

    response = llm.invoke(prompt)

    queries = [
        line.strip().lstrip("-").strip()
        for line in response.splitlines()
        if line.strip()
    ]

    return [question] + queries[:3]


# Test automated corrective query generation.
auto_queries = generate_corrective_queries(
    question,
    corrected_docs,
    llm
)

print("=" * 60)
print("AUTOMATED CORRECTIVE SEARCH QUERIES")
print("=" * 60)

for i, query in enumerate(auto_queries, 1):
    print(f"{i}. {query}")


AUTOMATED CORRECTIVE SEARCH QUERIES
1. What problem does eDySec aim to solve?
2. What are the limitations of existing malicious package detection methods that eDySec aims to address?
3. What are the challenges in interpreting model decisions in malicious package detection and how do they impact security?
4. How does the lack of transparency and explainability in existing dynamic security analysis frameworks create a "dangerous illusion of security"?


In [44]:

# ============================================================
# STEP 5I: EXECUTE AUTOMATED CORRECTIVE RETRIEVAL
# ============================================================

def automated_corrective_retrieval(
    question,
    search_queries,
    vectorstore,
    reranker,
    top_n=5
):
    all_docs = []

    # Retrieve evidence for each generated query.
    for query in search_queries:
        retrieved = vectorstore.similarity_search(
            query,
            k=6
        )
        all_docs.extend(retrieved)

    # Remove duplicate evidence chunks.
    unique_docs = []
    seen = set()

    for doc in all_docs:
        key = (
            doc.metadata.get("source"),
            doc.metadata.get("page"),
            doc.page_content
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    # Rank documents against the original question.
    ranked_docs = rerank_documents(
        question,
        unique_docs,
        reranker,
        top_n=top_n
    )

    return ranked_docs, unique_docs


auto_corrected_docs, auto_candidate_docs = (
    automated_corrective_retrieval(
        question,
        auto_queries,
        vectorstore,
        reranker,
        top_n=5
    )
)

print("=" * 60)
print("AUTOMATED CORRECTIVE RETRIEVAL RESULTS")
print("=" * 60)

print("Generated queries:", len(auto_queries))
print("Unique candidate chunks:", len(auto_candidate_docs))

for i, (doc, score) in enumerate(auto_corrected_docs, 1):
    print(f"\nRank {i}")
    print("Page:", doc.metadata.get("page_label"))
    print("Relevance score:", round(float(score), 4))
    print(doc.page_content[:500])


AUTOMATED CORRECTIVE RETRIEVAL RESULTS
Generated queries: 4
Unique candidate chunks: 18

Rank 1
Page: 2
Relevance score: 1.421
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per pack

Rank 2
Page: 2
Relevance score: 1.3178
ment, and transparent interpretation of model decisions, creates a
dangerous illusion of security and highlights the need for a more
comprehensive analytical framework.
To address these challenges, this study presents eDySec, an effi-
cient, stable, and explainable DL-basedextendedDynamicSecurity
analysis framework for detecting m

In [45]:

# ============================================================
# STEP 5 FIX: CORRECTED EVIDENCE SELECTION
# ============================================================

def build_corrected_evidence(
    question,
    original_docs,
    corrective_candidates,
    vectorstore,
    reranker,
    top_n=7
):
    # Include the original retrieved evidence.
    all_docs = [doc for doc, score in original_docs]

    # Include corrective retrieval candidates.
    all_docs.extend(corrective_candidates)

    # Retrieve additional evidence for the original question.
    all_docs.extend(
        vectorstore.similarity_search(question, k=10)
    )

    # Remove duplicate chunks.
    unique_docs = []
    seen = set()

    for doc in all_docs:
        key = (
            doc.metadata.get("source"),
            doc.metadata.get("page"),
            doc.page_content
        )

        if key not in seen:
            seen.add(key)
            unique_docs.append(doc)

    # Cross-Encoder reranking.
    ranked_docs = rerank_documents(
        question,
        unique_docs,
        reranker,
        top_n=len(unique_docs)
    )

    # Prefer evidence with positive relevance scores.
    # These are raw model scores, not probabilities.
    positive_docs = [
        (doc, score)
        for doc, score in ranked_docs
        if float(score) > 0
    ]

    # Fall back to top-ranked evidence if none scores positive.
    if not positive_docs:
        positive_docs = ranked_docs

    # Select the strongest evidence chunks.
    final_docs = positive_docs[:top_n]

    return final_docs


fixed_docs = build_corrected_evidence(
    question=question,
    original_docs=reranked_docs,
    corrective_candidates=auto_candidate_docs,
    vectorstore=vectorstore,
    reranker=reranker,
    top_n=7
)

print("=" * 60)
print("CORRECTED EVIDENCE SELECTION")
print("=" * 60)

for i, (doc, score) in enumerate(fixed_docs, 1):
    print(f"\nRank {i}")
    print("Page:", doc.metadata.get("page_label"))
    print("Score:", round(float(score), 4))
    print(doc.page_content[:500])


CORRECTED EVIDENCE SELECTION

Rank 1
Page: 2
Score: 1.421
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per pack

Rank 2
Page: 2
Score: 1.3178
ment, and transparent interpretation of model decisions, creates a
dangerous illusion of security and highlights the need for a more
comprehensive analytical framework.
To address these challenges, this study presents eDySec, an effi-
cient, stable, and explainable DL-basedextendedDynamicSecurity
analysis framework for detecting malicious packages. Figure 1 out-
lines the overall system architecture of the p

In [46]:

# ============================================================
# STEP 5: VERIFY CORRECTED EVIDENCE
# ============================================================

fixed_status = grade_retrieval(
    question,
    fixed_docs,
    llm
)

print("=" * 60)
print("FINAL EVIDENCE VERIFICATION")
print("=" * 60)

print("Question:", question)
print("Original status:", retrieval_status)
print("Corrected status:", fixed_status)

if fixed_status == "SUFFICIENT":
    print("Grader decision: Evidence is sufficient.")
    print("Next: Manually verify supporting passages.")
else:
    print("Grader decision: Evidence is still insufficient.")
    print("Next: Inspect grader output and evidence coverage.")


FINAL EVIDENCE VERIFICATION
Question: What problem does eDySec aim to solve?
Original status: INSUFFICIENT
Corrected status: INSUFFICIENT
Grader decision: Evidence is still insufficient.
Next: Inspect grader output and evidence coverage.


In [47]:

# ============================================================
# STEP 5K: IMPROVED EVIDENCE SUFFICIENCY GRADER
# ============================================================

import json
import re

def grade_evidence_v2(question, evidence_docs, llm):

    if not evidence_docs:
        return {
            "decision": "INSUFFICIENT",
            "reason": "No evidence retrieved.",
            "supporting_evidence": [],
            "raw_response": ""
        }

    context = "\n\n".join(
        f"[Evidence {i + 1} | Page "
        f"{doc.metadata.get('page_label', 'Unknown')}]\n"
        f"{doc.page_content}"
        for i, (doc, score) in enumerate(evidence_docs)
    )

    prompt = f"""
You are an evidence sufficiency evaluator for scientific
question answering.

QUESTION:
{question}

RETRIEVED EVIDENCE:
{context}

Evaluate whether the provided evidence is sufficient to
answer the question accurately.

Rules:
1. Use ONLY the retrieved evidence.
2. Judge whether the main question can be answered.
3. Do not require unrelated experimental details.
4. Evidence may be distributed across multiple passages.
5. Do not assume missing information.
6. If evidence supports only part of the required answer,
   return INSUFFICIENT.
7. Do not generate a full answer.

Return ONLY a valid JSON object:

{{
  "decision": "SUFFICIENT or INSUFFICIENT",
  "reason": "Brief explanation",
  "supporting_evidence": [1, 2]
}}
"""

    raw_response = llm.invoke(prompt).strip()

    try:
        # Allow JSON enclosed in a Markdown code fence.
        cleaned = re.sub(
            r"^```(?:json)?\\s*|\\s*```$",
            "",
            raw_response,
            flags=re.IGNORECASE
        ).strip()

        result = json.loads(cleaned)

        decision = str(
            result.get("decision", "")
        ).strip().upper()

        if decision not in ["SUFFICIENT", "INSUFFICIENT"]:
            decision = "UNKNOWN"

        supporting = result.get("supporting_evidence", [])

        if not isinstance(supporting, list):
            supporting = []

        valid_supporting = [
            i for i in supporting
            if type(i) is int
            and 1 <= i <= len(evidence_docs)
        ]

        return {
            "decision": decision,
            "reason": str(result.get("reason", "")),
            "supporting_evidence": valid_supporting,
            "raw_response": raw_response
        }

    except (json.JSONDecodeError, TypeError, ValueError):
        return {
            "decision": "UNKNOWN",
            "reason": "Unable to parse grader response.",
            "supporting_evidence": [],
            "raw_response": raw_response
        }


# Test the improved grader on the corrected evidence.
grade_result = grade_evidence_v2(
    question,
    fixed_docs,
    llm
)

print("=" * 60)
print("IMPROVED EVIDENCE SUFFICIENCY EVALUATION")
print("=" * 60)

print("Question:", question)
print("Decision:", grade_result["decision"])
print("Reason:", grade_result["reason"])
print(
    "Supporting evidence:",
    grade_result["supporting_evidence"]
)

if grade_result["decision"] == "UNKNOWN":
    print("\nRaw model response:")
    print(grade_result["raw_response"])


IMPROVED EVIDENCE SUFFICIENCY EVALUATION
Question: What problem does eDySec aim to solve?
Decision: SUFFICIENT
Reason: The evidence clearly states that eDySec aims to solve the problem of detecting malicious packages by providing an efficient, stable, and explainable framework for dynamic behavioral analysis.
Supporting evidence: [1, 2, 4]


# Grounded Answer Generation

In [48]:

# ============================================================
# STEP 6A: GROUNDED ANSWER GENERATION
# ============================================================

def generate_grounded_answer(question, evidence_docs, grade_result, llm):

    # Stop if evidence is insufficient or grading is uncertain.
    if grade_result["decision"] != "SUFFICIENT":
        return {
            "status": "ABSTAINED",
            "answer": "Insufficient verified evidence to answer.",
            "sources": []
        }

    # Build context with evidence IDs and page numbers.
    context_parts = []
    source_pages = set()

    for i, (doc, score) in enumerate(evidence_docs, 1):

        page = doc.metadata.get("page_label", "Unknown")

        context_parts.append(
            f"[Evidence {i} | Page {page}]\n"
            f"{doc.page_content}"
        )

        source_pages.add(str(page))

    context = "\n\n".join(context_parts)

    prompt = f"""
You are a scientific question-answering assistant.

Answer the question using ONLY the evidence provided.

QUESTION:
{question}

EVIDENCE:
{context}

STRICT RULES:
1. Use only information explicitly supported by the evidence.
2. Do not use external knowledge.
3. Do not invent facts, statistics, or explanations.
4. Explain the answer clearly and concisely.
5. Cite supporting evidence using [Evidence N, Page X].
6. Only cite evidence that actually supports the statement.
7. If the evidence cannot support an answer, respond:
   INSUFFICIENT EVIDENCE
8. Do not include unsupported claims.

ANSWER:
"""

    answer = llm.invoke(prompt).strip()

    if "INSUFFICIENT EVIDENCE" in answer.upper():
        return {
            "status": "ABSTAINED",
            "answer": "Insufficient evidence to answer.",
            "sources": []
        }

    return {
        "status": "GENERATED_UNVERIFIED",
        "answer": answer,
        "sources": sorted(source_pages)
    }


# Generate an answer using Step 5K evidence.
answer_result = generate_grounded_answer(
    question=question,
    evidence_docs=fixed_docs,
    grade_result=grade_result,
    llm=llm
)

print("=" * 60)
print("GROUNDED ANSWER GENERATION")
print("=" * 60)

print("Question:", question)
print("Status:", answer_result["status"])

print("\nGenerated Answer:")
print(answer_result["answer"])

print("\nRetrieved source pages:")
print(answer_result["sources"])


GROUNDED ANSWER GENERATION
Question: What problem does eDySec aim to solve?
Status: GENERATED_UNVERIFIED

Generated Answer:
Based on the evidence provided, the problem that eDySec aims to solve is:

**Malicious package detection in the software supply chain, specifically in the PyPI ecosystem.**

This is supported by the following evidence:

* [Evidence 1, Page 2]: "This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages."
* [Evidence 4, Page 1]: "malicious package detection. Additionally, model stability analysis and explainable AI techniques are incorporated into the detection pipeline..."
* [Evidence 5, Page 1]: "This paper presents eDySec, a DL-based efficient, stable, and explainable framework for dynamic behavioral analysis to detect malicious packages."

Furthermore, the evidence highlights the challenges of detecting malicious packages, including the high-dimensional and sparse natu

#Atomic Claim Extraction

In [49]:

# ============================================================
# STEP 7A: ATOMIC CLAIM EXTRACTION
# ============================================================

import json
import re

def extract_atomic_claims(answer, llm):

    prompt = f"""
You are a scientific claim extraction assistant.

Extract the factual claims from the answer below.

ANSWER:
{answer}

RULES:
1. Extract factual claims only.
2. Each claim must express one independently verifiable fact.
3. Split compound statements into separate claims.
4. Preserve the original meaning.
5. Do not introduce new information.
6. Ignore citation markers and introductory phrases.
7. Do not evaluate whether claims are true.
8. Return only a JSON array of strings.

Example:
[
  "The framework detects malicious packages.",
  "The framework uses dynamic behavioral analysis."
]

CLAIMS:
"""

    raw_response = llm.invoke(prompt).strip()

    # Remove optional Markdown code fences.
    cleaned = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        raw_response,
        flags=re.IGNORECASE
    ).strip()

    try:
        claims = json.loads(cleaned)

        if not isinstance(claims, list):
            raise ValueError("Expected a JSON array.")

        claims = [
            claim.strip()
            for claim in claims
            if isinstance(claim, str) and claim.strip()
        ]

        return claims

    except (json.JSONDecodeError, ValueError):
        print("Claim extraction could not be parsed.")
        print("Raw response:", raw_response)
        return []


# Extract claims from the generated answer.
atomic_claims = extract_atomic_claims(
    answer_result["answer"],
    llm
)

print("=" * 60)
print("ATOMIC CLAIM EXTRACTION")
print("=" * 60)

print("Number of claims:", len(atomic_claims))

for i, claim in enumerate(atomic_claims, 1):
    print(f"\nClaim {i}: {claim}")


Claim extraction could not be parsed.
Raw response: Here are the extracted factual claims in JSON format:

[
  "The problem that eDySec aims to solve is malicious package detection in the software supply chain.",
  "The problem that eDySec aims to solve is specifically in the PyPI ecosystem.",
  "eDySec is an efficient framework.",
  "eDySec is a stable framework.",
  "eDySec is an explainable framework.",
  "eDySec uses a DL-based approach.",
  "eDySec uses dynamic behavioral analysis.",
  "Dynamic behavioral data is high-dimensional.",
  "Dynamic behavioral data is sparse.",
  "A more comprehensive analytical framework is needed for malicious package detection."
]
ATOMIC CLAIM EXTRACTION
Number of claims: 0


In [50]:

# ============================================================
# STEP 7A FIX: ROBUST ATOMIC CLAIM EXTRACTION
# ============================================================

import json
import re

def extract_atomic_claims(answer, llm):

    prompt = f"""
You are a scientific claim extraction assistant.

Extract independently verifiable factual claims.

ANSWER:
{answer}

RULES:
1. Extract factual claims only.
2. Each claim must contain one verifiable fact.
3. Split compound statements.
4. Preserve the original meaning.
5. Do not invent new information.
6. Ignore citation markers and introductory phrases.
7. Do not evaluate whether claims are true.
8. Return a JSON array of strings.

CLAIMS:
"""

    raw_response = llm.invoke(prompt).strip()

    # Locate a JSON array even when the model adds extra text.
    decoder = json.JSONDecoder()
    claims = None

    for match in re.finditer(r"\[", raw_response):
        try:
            parsed, _ = decoder.raw_decode(
                raw_response[match.start():]
            )

            if (
                isinstance(parsed, list)
                and all(isinstance(item, str) for item in parsed)
            ):
                claims = parsed
                break

        except json.JSONDecodeError:
            continue

    if claims is None:
        print("ERROR: No valid JSON claim array found.")
        print("Raw response:", raw_response)
        return []

    # Remove empty and duplicate claims.
    cleaned_claims = []
    seen = set()

    for claim in claims:
        claim = claim.strip()
        key = claim.casefold()

        if claim and key not in seen:
            cleaned_claims.append(claim)
            seen.add(key)

    return cleaned_claims


# Run corrected claim extraction.
atomic_claims = extract_atomic_claims(
    answer_result["answer"],
    llm
)

print("=" * 60)
print("CORRECTED ATOMIC CLAIM EXTRACTION")
print("=" * 60)

print("Number of claims:", len(atomic_claims))

for i, claim in enumerate(atomic_claims, 1):
    print(f"\nClaim {i}: {claim}")


CORRECTED ATOMIC CLAIM EXTRACTION
Number of claims: 7

Claim 1: This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages.

Claim 2: malicious package detection is a problem that eDySec aims to solve.

Claim 3: malicious package detection is a challenge in the PyPI ecosystem.

Claim 4: dynamic behavioral data is high-dimensional and sparse.

Claim 5: a more comprehensive analytical framework is needed for detecting malicious packages.

Claim 6: model stability analysis and explainable AI techniques are incorporated into the detection pipeline.

Claim 7: eDySec is a DL-based efficient, stable, and explainable framework for dynamic behavioral analysis to detect malicious packages.


In [51]:

# ============================================================
# STEP 7B: CLAIM-LEVEL EVIDENCE VERIFICATION
# ============================================================

import json
import re

def parse_json_object(raw_response):
    decoder = json.JSONDecoder()

    for match in re.finditer(r"\{", raw_response):
        try:
            result, _ = decoder.raw_decode(
                raw_response[match.start():]
            )

            if isinstance(result, dict):
                return result

        except json.JSONDecodeError:
            continue

    return None


def verify_claim(claim, evidence_docs, llm):

    context = "\n\n".join(
        f"[Evidence {i} | Page "
        f"{doc.metadata.get('page_label', 'Unknown')}]\n"
        f"{doc.page_content}"
        for i, (doc, score) in enumerate(evidence_docs, 1)
    )

    prompt = f"""
You are a scientific evidence verification assistant.

CLAIM:
{claim}

RETRIEVED EVIDENCE:
{context}

TASK:
Determine whether the evidence supports or contradicts
the claim.

RULES:
1. Use ONLY the provided evidence.
2. SUPPORTED means the evidence directly supports the claim.
3. CONTRADICTED means the evidence explicitly conflicts
   with the claim.
4. INSUFFICIENT means neither support nor contradiction
   can be established.
5. Do not use external knowledge.
6. Do not assume a claim is false because evidence is missing.
7. Identify the evidence IDs supporting your decision.
8. Return only a JSON object.

OUTPUT FORMAT:
{{
    "decision": "SUPPORTED",
    "reason": "Brief evidence-based explanation",
    "evidence_ids": [1, 2]
}}
"""

    raw_response = llm.invoke(prompt).strip()
    result = parse_json_object(raw_response)

    if result is None:
        return {
            "decision": "UNKNOWN",
            "reason": "Unable to parse verification response.",
            "evidence_ids": [],
            "raw_response": raw_response
        }

    decision = str(
        result.get("decision", "")
    ).strip().upper()

    allowed = {
        "SUPPORTED",
        "CONTRADICTED",
        "INSUFFICIENT"
    }

    if decision not in allowed:
        decision = "UNKNOWN"

    evidence_ids = result.get("evidence_ids", [])

    if not isinstance(evidence_ids, list):
        evidence_ids = []

    evidence_ids = [
        i for i in evidence_ids
        if type(i) is int
        and 1 <= i <= len(evidence_docs)
    ]

    # A supported or contradicted claim must cite evidence.
    if decision in {"SUPPORTED", "CONTRADICTED"} and not evidence_ids:
        decision = "UNKNOWN"

    return {
        "decision": decision,
        "reason": str(result.get("reason", "")),
        "evidence_ids": evidence_ids,
        "raw_response": raw_response
    }


# Verify every extracted claim.
verification_results = []

print("=" * 60)
print("CLAIM-LEVEL EVIDENCE VERIFICATION")
print("=" * 60)

for i, claim in enumerate(atomic_claims, 1):

    result = verify_claim(
        claim,
        fixed_docs,
        llm
    )

    verification_results.append({
        "claim": claim,
        **result
    })

    print(f"\nClaim {i}: {claim}")
    print("Decision:", result["decision"])
    print("Reason:", result["reason"])
    print("Evidence IDs:", result["evidence_ids"])

    if result["decision"] == "UNKNOWN":
        print("Raw response:", result["raw_response"])


CLAIM-LEVEL EVIDENCE VERIFICATION

Claim 1: This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages.
Decision: SUPPORTED
Reason: The evidence directly supports the claim by presenting eDySec as an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages, with experimental results demonstrating its superiority over state-of-the-art frameworks.
Evidence IDs: [1, 2, 4, 5]

Claim 2: malicious package detection is a problem that eDySec aims to solve.
Decision: SUPPORTED
Reason: The evidence directly supports the claim by describing eDySec as a framework for detecting malicious packages and presenting experimental results demonstrating its effectiveness.
Evidence IDs: [1, 2, 4, 5]

Claim 3: malicious package detection is a challenge in the PyPI ecosystem.
Decision: SUPPORTED
Reason: The evidence directly supports the claim that mali

In [52]:

# ============================================================
# STEP 7C: VERIFICATION SUMMARY AND FINAL DECISION
# ============================================================

from collections import Counter

def summarize_verification(verification_results):

    counts = Counter(
        result["decision"]
        for result in verification_results
    )

    total = len(verification_results)
    supported = counts["SUPPORTED"]
    contradicted = counts["CONTRADICTED"]
    insufficient = counts["INSUFFICIENT"]
    unknown = counts["UNKNOWN"]

    if total == 0:
        final_decision = "ABSTAIN"

    elif contradicted > 0:
        final_decision = "REVISE"

    elif insufficient > 0 or unknown > 0:
        final_decision = "ABSTAIN"

    elif supported == total:
        final_decision = "ACCEPT_FOR_REVIEW"

    else:
        final_decision = "ABSTAIN"

    return {
        "total_claims": total,
        "supported": supported,
        "contradicted": contradicted,
        "insufficient": insufficient,
        "unknown": unknown,
        "support_rate": supported / total if total else 0.0,
        "final_decision": final_decision
    }


verification_summary = summarize_verification(
    verification_results
)

print("=" * 60)
print("CLAIM VERIFICATION SUMMARY")
print("=" * 60)

for key, value in verification_summary.items():
    print(f"{key}: {value}")


CLAIM VERIFICATION SUMMARY
total_claims: 7
supported: 7
contradicted: 0
insufficient: 0
unknown: 0
support_rate: 1.0
final_decision: ACCEPT_FOR_REVIEW


In [53]:

# ============================================================
# STEP 7D: INSPECT CLAIM-TO-EVIDENCE CITATIONS
# ============================================================

def inspect_claim_citations(verification_results, evidence_docs):

    print("=" * 60)
    print("CLAIM-TO-EVIDENCE CITATION INSPECTION")
    print("=" * 60)

    for claim_number, result in enumerate(
        verification_results, 1
    ):
        print(f"\n{'=' * 60}")
        print(f"CLAIM {claim_number}")
        print(result["claim"])
        print("Verifier decision:", result["decision"])

        evidence_ids = result.get("evidence_ids", [])

        if not evidence_ids:
            print("No supporting citations provided.")
            continue

        for evidence_id in evidence_ids:

            if not 1 <= evidence_id <= len(evidence_docs):
                print("Invalid evidence ID:", evidence_id)
                continue

            doc, score = evidence_docs[evidence_id - 1]

            print(f"\n  Evidence {evidence_id}")
            print("  Page:", doc.metadata.get("page_label"))
            print("  Passage:")
            print(doc.page_content[:600])


inspect_claim_citations(
    verification_results,
    fixed_docs
)


CLAIM-TO-EVIDENCE CITATION INSPECTION

CLAIM 1
This study presents eDySec, an efficient, stable, and explainable DL-based extended Dynamic Security analysis framework for detecting malicious packages.
Verifier decision: SUPPORTED

  Evidence 1
  Page: 2
  Passage:
techniques into the detection pipeline, enabling near-perfect
stable and transparent interpretation of model decisions by
identifying the features that most influence each prediction.
• Experimental results demonstrate that eDySec significantly
outperforms the state-of-the-art framework. Specifically, it
reduces feature dimensionality by 52.78% while lowering
false positives by 82% and false negatives by 79%. It also
improves accuracy by 3% and maintains a low inference
latency of 170ms per package.
This study is structured as follows: Section 2 reviews existing
methods for malicious package d

  Evidence 2
  Page: 2
  Passage:
ment, and transparent interpretation of model decisions, creates a
dangerous illusion of security a